# Perspect: Propaganda Detection Transformer Model

## Data Preprocessing

Initial data loading, conversion of label strings to lists, and downsampling of 'None' labels were performed to balance the dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch scikit-learn
import pandas as pd, ast, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset, DataLoader

train_df = pd.read_csv("/content/drive/MyDrive/AI4Good/V4/Final Data/train_all_languages.csv")
dev_df = pd.read_csv("/content/drive/MyDrive/AI4Good/V4/Final Data/dev_all_languages.csv")

train_df["labels"] = train_df["labels"].apply(ast.literal_eval)
dev_df["labels"] = dev_df["labels"].apply(ast.literal_eval)

none_rows = train_df[train_df["labels"].apply(lambda x: x == ["None"])]
keep = none_rows.sample(frac=0.5, random_state=42)
non_none_rows = train_df[train_df["labels"].apply(lambda x: x != ["None"])]
train_df = pd.concat([non_none_rows, keep]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"New train size after 'None' downsampling: {len(train_df)}")

## Label Binarization & Tokenization

Labels were converted to a binary matrix format using MultiLabelBinarizer, and text data was tokenized using roberta-base.

In [ ]:
mlb = MultiLabelBinarizer()
train_labels = mlb.fit_transform(train_df["labels"])
dev_labels = mlb.transform(dev_df["labels"])  # transform only, not fit

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(texts):
    return tokenizer(list(texts), max_length=256, padding="max_length", truncation=True, return_tensors="pt")

train_enc = tokenize(train_df["text"])
dev_enc = tokenize(dev_df["text"])

class PropDataset(Dataset):
    def __init__(self, enc, labels):
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __getitem__(self, idx):
        return {"input_ids": self.input_ids[idx], "attention_mask": self.attention_mask[idx], "labels": self.labels[idx]}

    def __len__(self):
        return len(self.labels)

train_ds = PropDataset(train_enc, train_labels)
dev_ds = PropDataset(dev_enc, dev_labels)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=16)

print(f"Initial number of labels: {len(mlb.classes_)}")

## Model Initialization

A `roberta-base` model for sequence classification was loaded, with `num_labels` initially set based on the original label set.

In [ ]:
num_labels = len(mlb.classes_)
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=num_labels, problem_type="multi_label_classification")
print(f"Model initialized with {num_labels} labels.")

## Class Imbalance Handling

`pos_weight_tensor` was calculated for each class to address imbalances and was incorporated into a `CustomTrainer` class, which uses `BCEWithLogitsLoss`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import Trainer

num_positive_samples = train_labels.sum(axis=0)
num_negative_samples = train_labels.shape[0] - num_positive_samples

raw_weights = num_negative_samples / np.clip(num_positive_samples, 1, None)
pos_weight_tensor = torch.tensor(
    np.log1p(raw_weights),  # log(1 + ratio) compresses extreme values naturally
    dtype=torch.float
)

print("Initial pos_weight_tensor per Class (first iteration):")
for cls, w in zip(mlb.classes_, pos_weight_tensor):
    print(f"{cls:<40} {w:.2f}")

class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## Training Parameters

The model was configured with `TrainingArguments` including 5 training epochs, a `weight_decay` of 0.01, and `EarlyStoppingCallback`.

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score

output_dir = "/content/drive/.shortcut-targets-by-id/1j5UloTTUCyec07AeiAZSpn0DLTFjmosq/V4/models/roberta_propaganda_classifier"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="tensorboard",
    logging_steps=100,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="f1_macro",
    load_best_model_at_end=True,
    greater_is_better=True,
    save_total_limit=1
)

def compute_metrics(p):
    predictions = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    labels = p.label_ids

    probs = 1 / (1 + np.exp(-predictions))
    binary_predictions = (probs > 0.5).astype(int)

    f1_micro = f1_score(labels, binary_predictions, average='micro')
    f1_macro = f1_score(labels, binary_predictions, average='macro')
    accuracy = accuracy_score(labels, binary_predictions)

    return {"f1_micro": f1_micro, "f1_macro": f1_macro, "accuracy": accuracy}

## Data Filtering (First Iteration)

Problematic detailed labels (e.g., 'Whataboutism', 'Appeal_to_Time') were removed, reducing the number of labels to 16. The `MultiLabelBinarizer`, `num_labels`, and `pos_weight_tensor` were re-calculated, and the model was re-initialized and re-trained (implicitly for setting up the subsequent step).

In [ ]:
# Define classes to drop based on analysis
classes_to_drop = [
    'Whataboutism', 'Appeal_to_Time', 'Straw_Man', 'Red_Herring',
    'Causal_Oversimplification', 'Appeal_to_Popularity',
    'Obfuscation-Vagueness-Confusion', 'Repetition'
]

print("--- Data Filtering and Re-preparation (First Iteration) ---")
print(f"Original train_df shape: {train_df.shape}")
print(f"Original dev_df shape: {dev_df.shape}")

def filter_dataframe_by_labels(df, classes_to_drop):
    mask = df['labels'].apply(lambda label_list: not any(label in classes_to_drop for label in label_list))
    return df[mask].reset_index(drop=True)

train_df = filter_dataframe_by_labels(train_df, classes_to_drop)
dev_df = filter_dataframe_by_labels(dev_df, classes_to_drop)

print(f"Filtered train_df shape: {train_df.shape}")
print(f"Filtered dev_df shape: {dev_df.shape}")

# Re-initialize MultiLabelBinarizer with only the labels present in the filtered training data
all_remaining_labels_in_train = set()
for label_list in train_df["labels"]:
    for label in label_list:
        all_remaining_labels_in_train.add(label)

remaining_classes_sorted = sorted(list(all_remaining_labels_in_train))
mlb = MultiLabelBinarizer(classes=remaining_classes_sorted)
mlb.fit(train_df["labels"])

train_labels = mlb.transform(train_df["labels"])
dev_labels = mlb.transform(dev_df["labels"])

num_labels = len(mlb.classes_)

print(f"New num_labels after first filtering: {num_labels}")

# Re-tokenize the filtered dataframes and re-create datasets/dataloaders
train_enc = tokenize(train_df["text"])
dev_enc = tokenize(dev_df["text"])

train_ds = PropDataset(train_enc, train_labels)
dev_ds = PropDataset(dev_enc, dev_labels)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=16)

# Re-calculate pos_weight_tensor based on the new train_labels (16 labels)
num_positive_samples = train_labels.sum(axis=0)
num_negative_samples = train_labels.shape[0] - num_positive_samples

raw_weights = num_negative_samples / np.clip(num_positive_samples, 1, None)
pos_weight_tensor = torch.tensor(
    np.log1p(raw_weights),
    dtype=torch.float
)

print("New pos_weight_tensor per Class (first iteration filter):")
for cls, w in zip(mlb.classes_, pos_weight_tensor):
    print(f"{cls:<40} {w:.2f}")
print("Data re-preparation after first filtering complete.")

## Data Categorization (Second Iteration)

The remaining 16 detailed labels were mapped to 6 broader, general categories (e.g., 'Justification', 'Reputation'). The `MultiLabelBinarizer`, `num_labels` (now 6), and `pos_weight_tensor` were again re-calculated based on these new categories.

In [ ]:
# Define the mapping from detailed labels to general categories
general_category_mapping = {
    'Appeal_to_Authority': 'Justification',
    'Appeal_to_Values': 'Justification',
    'Appeal_to_Fear-Prejudice': 'Justification',
    'Flag_Waving': 'Justification',

    'False_Dilemma-No_Choice': 'Simplification',
    'Consequential_Oversimplification': 'Simplification',

    'Name_Calling-Labeling': 'Reputation',
    'Doubt': 'Reputation',
    'Guilt_by_Association': 'Reputation',
    'Appeal_to_Hypocrisy': 'Reputation',
    'Questioning_the_Reputation': 'Reputation',

    'Loaded_Language': 'Manipulative Wording',
    'Exaggeration-Minimisation': 'Manipulative Wording',

    'Slogans': 'Call',
    'Conversation_Killer': 'Call',

    'None': 'None'
}

def map_to_general_categories(detailed_labels_list):
    general_categories = set()
    for label in detailed_labels_list:
        if label in general_category_mapping:
            general_categories.add(general_category_mapping[label])
    return list(general_categories) if general_categories else []

print("--- Data Categorization (Second Iteration) ---")
# Overwrite the 'labels' column with the general categories
train_df['labels'] = train_df['labels'].apply(map_to_general_categories)
dev_df['labels'] = dev_df['labels'].apply(map_to_general_categories)

# Re-initialize MultiLabelBinarizer for General Categories
all_general_categories_in_train = set()
for label_list in train_df["labels"]:
    for label in label_list:
        all_general_categories_in_train.add(label)

final_general_classes_sorted = sorted(list(all_general_categories_in_train))
mlb = MultiLabelBinarizer(classes=final_general_classes_sorted)
mlb.fit(train_df["labels"])

print(f"MultiLabelBinarizer re-initialized with general categories: {mlb.classes_}")

train_labels = mlb.transform(train_df["labels"])
dev_labels = mlb.transform(dev_df["labels"])

num_labels = len(mlb.classes_)
print(f"New num_labels (general categories): {num_labels}")

# Re-calculate pos_weight_tensor for 6 general categories
num_positive_samples = train_labels.sum(axis=0)
num_negative_samples = train_labels.shape[0] - num_positive_samples

raw_weights = num_negative_samples / np.clip(num_positive_samples, 1, None)
pos_weight_tensor = torch.tensor(
    np.log1p(raw_weights),
    dtype=torch.float
)

print("New pos_weight_tensor per General Class:")
for cls, w in zip(mlb.classes_, pos_weight_tensor):
    print(f"{cls:<25} {w:.2f}")

# Re-create Datasets and Dataloaders with new labels
train_enc = tokenize(train_df["text"])
dev_enc = tokenize(dev_df["text"])

train_ds = PropDataset(train_enc, train_labels)
dev_ds = PropDataset(dev_enc, dev_labels)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=16)

print("Data re-preparation for 6 general categories complete.")

## Final Model Retraining

The RoBERTa model was re-initialized with `num_labels=6`, and the `CustomTrainer` was used to perform the final training run with the updated `pos_weight_tensor` and the dataset prepared for these 6 general categories. This final training resulted in the displayed `eval_f1_micro` of 0.6318 and `eval_f1_macro` of 0.5143.

In [ ]:
from transformers import AutoModelForSequenceClassification

# Re-initialize the model with the new num_labels (6)
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print(f"Model re-initialized with new num_labels for general categories: {num_labels}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
pos_weight_tensor = pos_weight_tensor.to(device)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("Starting re-training with custom loss and updated pos_weight for 6 general categories...")
trainer.train()
print("Re-training complete for 6 general categories.")

print("Evaluating re-trained model for 6 general categories...")
eval_results_general = trainer.evaluate()
print(f"Evaluation results (6 general categories): {eval_results_general}")

# print(f"Saving re-trained model (6 general categories) to {output_dir}...")
# trainer.save_model(output_dir)
# tokenizer.save_pretrained(output_dir)
# print("Re-trained model (6 general categories) and tokenizer saved.")

predictions_general = trainer.predict(dev_ds)
logits_general = predictions_general.predictions
probs_general = 1 / (1 + np.exp(-logits_general))
print("Mean prob (re-trained model, 6 general categories):", probs_general.mean())
print("Max prob (re-trained model, 6 general categories):", probs_general.max())
print("Min prob (re-trained model, 6 general categories):", probs_general.min())

## Threshold Tuning

Calculates the F1 score across a range of thresholds and selects the one that maximizes the F1 score.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def find_optimal_thresholds(true_labels, predicted_logits, num_classes, num_thresholds=100):
    optimal_thresholds = np.zeros(num_classes)
    probabilities = 1 / (1 + np.exp(-predicted_logits))

    for i in range(num_classes):
        best_f1 = -1
        best_threshold = 0.5
        thresholds = np.linspace(0.01, 0.99, num_thresholds)

        for t in thresholds:
            binary_predictions = (probabilities[:, i] > t).astype(int)
            f1 = f1_score(true_labels[:, i], binary_predictions, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_threshold = t
        optimal_thresholds[i] = best_threshold
    return optimal_thresholds

def analyze_per_class_f1_with_tuned_thresholds(true_labels, predicted_logits, class_names, thresholds):
    probabilities = 1 / (1 + np.exp(-predicted_logits))
    per_class_f1 = []
    for i, class_name in enumerate(class_names):
        binary_predictions = (probabilities[:, i] > thresholds[i]).astype(int)
        f1 = f1_score(true_labels[:, i], binary_predictions, zero_division=0)
        per_class_f1.append({'Class': class_name, 'F1 Score': f1})
    return pd.DataFrame(per_class_f1).sort_values('F1 Score', ascending=False).reset_index(drop=True)

optimal_thresholds_general = find_optimal_thresholds(dev_labels, logits_general, num_labels)

print("Optimal Thresholds per General Category:")
for i, class_name in enumerate(mlb.classes_):
    print(f"{class_name:<25} {optimal_thresholds_general[i]:.2f}")

per_class_f1_tuned_df_general = analyze_per_class_f1_with_tuned_thresholds(
    dev_labels, logits_general, mlb.classes_, optimal_thresholds_general
)

print("\nPer-Class F1 Scores (Development Set) with Tuned Thresholds (6 General Categories):")
display(per_class_f1_tuned_df_general)

plt.figure(figsize=(12, 6))
sns.barplot(x='F1 Score', y='Class', hue='Class', data=per_class_f1_tuned_df_general, palette='viridis', legend=False)
plt.title('Per-Class F1 Scores on Development Set with Tuned Thresholds (6 General Categories)')
plt.xlabel('F1 Score')
plt.ylabel('General Category Label')
plt.tight_layout()
plt.show()

print(f"Overall F1 Micro (6 General Categories): {eval_results_general['eval_f1_micro']:.4f}")
print(f"Overall F1 Macro (6 General Categories): {eval_results_general['eval_f1_macro']:.4f}")

print("\nMapping of numerical labels to general category names:")
for i, class_name in enumerate(mlb.classes_):
    print(f"Label {i}: {class_name}")

## Testing with a Synthetically Generated Test Set

In [ ]:
import pandas as pd
import ast

# @title Enter the path to your synthetic test set CSV file
test_csv_path = "/content/drive/MyDrive/AI4Good/V4/Data/synthetic test data - perspect.csv" # @param {type:"string"}

if not test_csv_path:
    print("Please provide a path to the test CSV file.")
else:
    print(f"Loading test data from: {test_csv_path}")
    synthetic_test_df = pd.read_csv(test_csv_path, header=None) # Load without header initially

    # It appears the actual column names are in the second row (index 1) of the CSV,
    # and the first row (index 0) might be empty or contain non-header info.

    # Extract column names from the second row and clean them
    header_row_values = [str(val).strip() for val in synthetic_test_df.iloc[1]]
    synthetic_test_df.columns = header_row_values

    # Drop the first two rows (the actual 'empty' row and the one used for headers)
    synthetic_test_df = synthetic_test_df[2:].reset_index(drop=True)

    # Ensure all column names are stripped of whitespace again, just in case.
    synthetic_test_df.columns = synthetic_test_df.columns.astype(str).str.strip()

    # As per user instruction, only consider 'text' and 'labels' columns
    # Ensure these columns exist before trying to select them
    if 'text' in synthetic_test_df.columns and 'labels' in synthetic_test_df.columns:
        synthetic_test_df = synthetic_test_df[['text', 'labels']]
    else:
        print("Error: After processing, 'text' or 'labels' columns were not found.")
        print("Available columns:", synthetic_test_df.columns.tolist())
        synthetic_test_df = pd.DataFrame() # Create an empty DataFrame to prevent further errors

    print("Columns in synthetic_test_df after processing:", synthetic_test_df.columns.tolist())
    print("Synthetic test DataFrame head after processing:\n", synthetic_test_df.head())

    # Check for required columns and proceed with processing
    required_columns = ['text', 'labels'] # Assuming 'text' and 'labels' are the desired names
    missing_columns = [col for col in required_columns if col not in synthetic_test_df.columns]

    if missing_columns:
        print(f"Error: Missing expected columns in synthetic test CSV: {missing_columns}.\n" +
              "Please check your CSV file's column names and update the 'required_columns' list in the code if they are different.")
    else:
        try:
            # Fill NaN values in 'labels' column with a string representation of ['None']
            # This ensures ast.literal_eval receives a string, even for previously NaN values.
            synthetic_test_df["labels"] = synthetic_test_df["labels"].fillna("['None']")
            synthetic_test_df["labels"] = synthetic_test_df["labels"].apply(ast.literal_eval)

            # Fill any NaN values in the 'text' column with an empty string
            synthetic_test_df['text'] = synthetic_test_df['text'].fillna('')

            # Filter out rows where 'text' is an empty string after filling NaNs
            synthetic_test_df = synthetic_test_df[synthetic_test_df['text'] != ''].reset_index(drop=True)
            print(f"Synthetic test DataFrame shape after removing empty text rows: {synthetic_test_df.shape}")

            # Apply the same filtering as to train_df and dev_df
            print("Applying first iteration filtering to test data...")
            synthetic_test_df = filter_dataframe_by_labels(synthetic_test_df, classes_to_drop)

            # Apply the same categorization as to train_df and dev_df
            print("Applying second iteration categorization to test data...")
            synthetic_test_df['labels'] = synthetic_test_df['labels'].apply(map_to_general_categories)

            # Transform labels using the existing MultiLabelBinarizer
            print("Transforming labels using existing MultiLabelBinarizer...")
            synthetic_test_labels = mlb.transform(synthetic_test_df["labels"])

            # Tokenize text using the existing tokenizer
            print("Tokenizing text...")
            synthetic_test_enc = tokenize(synthetic_test_df["text"])

            # Create a PropDataset for the synthetic test data
            synthetic_test_ds = PropDataset(synthetic_test_enc, synthetic_test_labels)

            print("Synthetic test data prepared successfully.")
            print(f"Synthetic test DataFrame head:\n{synthetic_test_df.head()}")

        except KeyError as e:
            print(f"A KeyError occurred during label processing: {e}.\nThis likely means the column name was not 'labels' or 'text' at an unexpected stage.")
        except Exception as e:
            print(f"An unexpected error occurred during processing: {e}")

In [ ]:
print("Train DataFrame head after all filtering and categorization:")
display(train_df.head())

print("\nDev DataFrame head after all filtering and categorization:")
display(dev_df.head())

In [ ]:
from collections import Counter

# Function to get label counts
def get_label_counts(df):
    all_labels = []
    for label_list in df['labels']:
        all_labels.extend(label_list)
    return Counter(all_labels)

print("\nTrain DataFrame Label Counts:")
train_label_counts = get_label_counts(train_df)
for label, count in train_label_counts.most_common():
    print(f"  {label}: {count}")

print("\nDev DataFrame Label Counts:")
dev_label_counts = get_label_counts(dev_df)
for label, count in dev_label_counts.most_common():
    print(f"  {label}: {count}")